In [ ]:
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")
print(jax.devices())
import jaxlib
import jax.numpy as jnp
import flax
import flax.linen as nn
import optax
from typing import Tuple, Callable, Any, Dict, Optional
import numpy.typing as npt
import copy
import pathlib
import matplotlib.pyplot as plt
import time
import json
import ast
import netket as nk
import os
import glob
import sys

sys.path.append("/home/ihuarte/Escritorio/Ivan/NNs")

# os.chdir("/home/ihuarte/Escritorio/Ivan/NN/")

from VA_project.model.model import J1J2Square
from VA_project.engine.runners import Runner

# from NN_utils import load_vstate
# from correlations import correlations_vstate

# from NNs.NN_module.ST_utils import compare_params, masked_optimizer
from frozendict import deepfreeze
from NN_module.ST_utils import print_tree

In [1]:
%run ./VMC_simulation.py

[CudaDevice(id=0)]
Configurations:
 - /home/ihuarte/Escritorio/Ivan/NNs/config.json
 - /home/ihuarte/Escritorio/Ivan/NNs/config_CM.json
 - /home/ihuarte/Escritorio/Ivan/NNs/config_Hydra.json
 - /home/ihuarte/Escritorio/Ivan/NNs/config_Hydra_NN.json
Running exact diagonalization...
Energy ED: -8.45792335139484


──────────────────────────────────── 🧲 CM: J1J2Square   🧠 NN: testAnsatz_CNN ────────────────────────────────────

🧱 Size: 4x4

╭────────────────────────────────────────────────── SIMULATION ───────────────────────────────────────────────────╮
│ ╭─────────────── Sampler ───────────────╮  ╭────────────── Schedule ───────────────╮                            │
│ │ rules=             prules= [0.2, 0.7, │  │ learning_rate=            sampler= {} │                            │
│ │ ['LocalRule',      0.1]               │  │ {'epochs_struct':                     │                            │
│ │ 'Exchange',                           │  │ [[[200]], [[1000]]],                  │                            │
│ │ 'InvertMagnetiza…                     │  │ 'modes_stru...                        │                            │
│ │ n_ranks= 1         n_chains_per_rank= │  ╰───────────────────────────────────────╯                            │
│ │                    32                 │                                                                       │
│ │ n_samples_per_ch…  chunk_vstate= 512  │                                                                       │
│ │ 64                                    │                                                                       │
│ ╰───────────────────────────────────────╯                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────── COUPLING MODEL (J1J2Square) ──────────────────────────────────────────╮
│ params= {'J2': 0.5, 'J1': 1.0, 'fields':       kwargs_lattice= {'bc': 'periodic', 'order':    S_operators= True │
│ [0.0, 0.0, 0.0]}                               'default_2'}                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── NEURAL NETWORK (testAnsatz_CNN) ────────────────────────────────────────╮
│ name= testAnsatz_CNN  setup= {'stage_0': {'template': {'Transversal': {'Factori...                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────────────────  ─────────────────────────────────────────────────────────

********************* ARCHITECTURE INFO *********************

└── Trans_0   --->  (0)
    └── phi   --->  (00)


NN stats: 16 parameters (0.0001220703125 MB)

*************************************************************
Total samples: 2048

Initializing sampler...
Initializing Variational State...
Adding BestIterKeeper
submod: {'0': ('Trans_0',)}, modes: ['A'], lr:[0.1]
submod: {'0': ('Trans_0',)}, modes: [('Trans_0',)], lr:[0.1]

Period 1 / 2:
Training ('Trans_0',) for 200 epochs
LR: 0.1    ()
Diagonal shift: 1.0000e-02

[('Trans_0',)]
Trans_0
   phi: train_0
None

Training ('Trans_0',) for 200 epochs...


  0%|          | 0/200 [00:00<?, ?it/s]

VS phase: 12.480455359699533 ± 4.410896377691281  (complex)
Changing Architecture


In [6]:
from NN_module.NN.utils import (
    setup_from_template,
    get_code2path_tree,
    print_architecture,
    get_code2path_flatten,
    change_module_attr,
    greedy_transplant,
    get_subtree,
    set_subtree,
)


In [ ]:
hydra.arch_evol(vstate.parameters)


Saving stage_0 parameters....
Saved.


AttributeError: 'tuple' object has no attribute 'keys'

In [5]:
hydra.template

{'SplitTraining': {'CNN': {}, 'Factorized': {}}}

In [9]:
stage_config = hydra.arch_evolution[f"stage_{hydra.n_stage}"]
raw_setup = setup_from_template(hydra.template, hydra.storage, hydra.symm_wrapper)
if "change_attr" in stage_config:
    if stage_config["change_attr"]:
        raw_setup = change_module_attr(raw_setup, stage_config["change_attr"])


In [11]:
raw_setup = raw_setup[0]

In [12]:
raw_setup

{'module': 'SplitTraining',
 'setup': {'modulus': {'module': 'CNN',
   'setup': {'channels': [32, 16, 4],
    'strides': [[1, 1], [1, 1], [1, 1]],
    'kernel': [3, 3],
    'use_pooling': False,
    'pooling_strides': [[1, 1], [1, 1]],
    'final_architecture': [1]}},
  'phase': {'module': 'Factorized',
   'setup': {'complex': True, 'lattice_size': [4, 4]}},
  'symm_Z2': False,
  'trivial_Z2': True,
  'symm_2D': False,
  'irrep': [0, 0],
  'use_anchor': True,
  'squeeze': True}}

In [61]:
def change_values(module_setup, changes):
    for attr, value in changes.items():
        if attr != "_count":
            module_setup[attr] = value
    return module_setup

from NN_module.NN import __all_single__
def change_module_attr(setup, changes):

    new_setup = {}
    if all([key in ["module","setup"] for key in setup.keys()]): 
        new_setup["module"] = setup["module"]
        if setup["module"] in changes.keys(): # Then is a change_target
            

            if "_count" in changes[setup["module"]]:
                if changes[setup["module"]]["_count"] != 0:
                    changes[setup["module"]]["_count"] += -1
                    new_setup["setup"] = setup["setup"]

                else:
                    new_setup["setup"] = change_values(setup["setup"], changes[setup["module"]])


            else:
                new_setup["setup"] = change_values(setup["setup"], changes[setup["module"]])
        
        elif setup["module"] in __all_single__: # Then is another simple module
            new_setup["setup"] = setup["setup"]


        else:
            new_setup["setup"], changes = change_module_attr(setup["setup"], changes)

    else: # module, phase, Seq, Trans
        for k, v in setup.items():
            if isinstance(v, dict):
                new_setup[k], changes = change_module_attr(v, changes)
            else:
                new_setup[k] = v

    return new_setup, changes

In [65]:
from NN_module.ST_utils import print_tree
change = {
    "Factorized":{
        "_count": 0,
        "complex": 'ewsgefws'
    }
}
setup_2, _ = change_module_attr(raw_setup, change)

In [66]:
print_tree(setup_2, values=True)

module: SplitTraining
setup
   modulus
      module: CNN
      setup
         channels: [32, 16, 4]
         strides: [[1, 1], [1, 1], [1, 1]]
         kernel: [3, 3]
         use_pooling: False
         pooling_strides: [[1, 1], [1, 1]]
         final_architecture: [1]
   phase
      module: Factorized
      setup
         complex: ewsgefws
         lattice_size: [4, 4]
   symm_Z2: False
   trivial_Z2: True
   symm_2D: False
   irrep: [0, 0]
   use_anchor: True
   squeeze: True


In [54]:
print_tree(raw_setup, values=True)

module: SplitTraining
setup
   modulus
      module: CNN
      setup
         channels: [32, 16, 4]
         strides: [[1, 1], [1, 1], [1, 1]]
         kernel: [3, 3]
         use_pooling: False
         pooling_strides: [[1, 1], [1, 1]]
         final_architecture: [1]
   phase
      module: Factorized
      setup
         complex: seeee
         lattice_size: [4, 4]
   symm_Z2: False
   trivial_Z2: True
   symm_2D: False
   irrep: [0, 0]
   use_anchor: True
   squeeze: True


In [ ]:
from NN_module.schedule.schedule import Schedule
import optax

setup = {
    "print_arch": True,
    "learning_rate": {
        "epochs_struct": [[[50], [100], [200]]],
        "modes_struct": [[[["A"]], [["1", "110"]], [["10000"]]]],
        "lr_struct": [
            [[[0.01, 0.05]], [["lin(0.001, 0.1)", 0.3]], [["exp(0.1, 0.001)"]]]
        ],
        "repeat": [],
        "rescale": 1.0,
    },
    "architecture": {},
    "sampler": {},
}


sch = Schedule(setup, vstate.parameters)
sch.total_epochs
opt = optax.sgd
for period, info in sch.schedule():
    print(f"{info}")
    print(f"{period} {type(period)}\n")
    sch.transform_optimizer(vstate.parameters, opt, info, period)
    print("\n")

In [ ]:
import flax.linen as nn

arch_name = type(model).__name__
subarch_names = [
    name
    for name in model.__dict__.keys()
    if isinstance(model.__dict__[name], nn.Module)
]
arch_name, subarch_names

In [ ]:
model.__dict__["Trans"][0].__class__.__name__

In [ ]:
def is_subsequence(a, b):
    n, m = len(a), len(b)
    for i in range(m - n + 1):
        if b[i : i + n] == a:
            return True
    return False


a = (1, 2, 3)
b = (1, 2, 4, 3, 4, 5, 6)
is_subsequence(a, b)

In [ ]:
factory.setup